In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import os
import h5py
os.environ['PROJ_LIB'] = '/home/lty/anaconda3/envs/hloc/share/proj'

from osgeo import gdal, osr
from geopy.distance import geodesic

# from scipy.stats import pairs

from hloc import extract_features, match_features
from hloc.utils.io import get_matches, get_keypoints
from hloc.visualization import plot_images, plot_keypoints, plot_matches, read_image, add_text

from my_pkg.tools import sort_key, pixel_to_geo_coordinates, read_pairs, extract_rotation_angle, \
    draw_fusion_keyframe_traj, draw_keyframe_elevation_trajectory


In [3]:
image_dir = Path("/home/lty/datasets/RealUAV/city1/")
seu_uav_dir = image_dir / "uav"
seu_tif_dir = image_dir / "tif"
output_dir = Path("/home/lty/outputs/RealUAV/city1")
output_dir.mkdir(exist_ok=True, parents= True)
pairs_path = output_dir/"pairs.txt"
img_list_path = output_dir/"img_list.txt"
uav_list_path = output_dir/"uav_list.txt"
tif_list_path = output_dir/"tif_list.txt"
features_path = output_dir/"features.h5"
matches_path = output_dir/"matches.h5"
loc_path = output_dir/"loc.txt"# 保存定位结果

In [10]:
image_extensions = ['.jpg', '.jpeg', '.png', '.tif', '.tiff']
# 收集 'seu_uav' 文件夹中的所有图像文件
uav_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_uav_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

# 收集 'seu_tif' 文件夹中的所有图像文件
tif_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_tif_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]
uav_images = sorted(uav_images, key = sort_key)
tif_images = sorted(tif_images, key = sort_key)
# 将 UAV 和 TIF 图像文件列表合并
img_list = uav_images + tif_images
# 打印 img_list 以验证结果
# for img in img_list:
#     print(img)
#save img_list
with open(img_list_path, 'w') as f:
    for img in img_list:
        f.write(img + "\n")

with open(uav_list_path, 'w') as f:
    for img in uav_images:
        f.write(img + "\n")
with open(tif_list_path, 'w') as f:
    for img in tif_images:
        f.write(img + "\n")

In [11]:
import time
t0 = time.time()
extract_features.main(
    conf=extract_features.confs['superpoint_max'],
    image_dir=image_dir,
    image_list=img_list,
    feature_path = features_path,
)
t1 = time.time()
match_features.main(
    conf=match_features.confs["superpoint+lightglue"],
    pairs=pairs_path,
    features=features_path,
    matches=matches_path,
)
t2 = time.time()
print(f"Feature extraction time: {t1-t0:.3f}s")
print(f"Feature matching time: {t2-t1:.3f}s")

[2025/05/21 12:40:40 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2025/05/21 12:40:40 hloc INFO] Skipping the extraction.
[2025/05/21 12:40:40 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
[2025/05/21 12:40:40 hloc INFO] Skipping the matching.


Feature extraction time: 0.190s
Feature matching time: 0.008s


In [6]:
geotransform = []
# 打开 geotransform.txt 文件
with open("/home/lty/datasets/RealUAV/city1/geotransform.txt", "r") as f:
    # 逐行读取文件内容
    for line in f:
        # 去除行首尾的空白字符和换行符
        line = line.strip()
        if line:
            try:
                # 将字符串转换为浮点数
                value = float(line)
                # 将数值添加到列表中
                geotransform.append(value)
            except ValueError:
                print(f"无法将以下内容转换为数值：'{line}'")
                # 根据需要，可以选择跳过或停止程序
                continue
# 输出读取到的 geotransform 数据
print("Geotransform 数组：\n", geotransform)

Geotransform 数组：
 [12123218.434142068, 0.2985821417389691, 0.0, 4062398.742272254, 0.0, -0.2985821417389691]


In [45]:

pairs = read_pairs(pairs_path)
print(f"Found {len(pairs)} image pairs.")
start_x, start_y = 0, 0
x_in_map, y_in_map = 0, 0
x_origin, y_origin = 0, 0
start = time.time()
n = 0
# match_save_path = output_dir/"matches"
with open(loc_path, 'w') as loc_file:
    for img_uav, img_tif in pairs:
        print(f"UAV: {img_uav} - TIF: {img_tif}")
        kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
        matches,scores = get_matches(matches_path, img_uav, img_tif)
        print(matches.shape)
        pts1 = kp1[matches[:,0]]
        pts2 = kp2[matches[:,1]]
        
        H, mask = cv2.findHomography(pts1, pts2, cv2.RHO, 10.0)

        print(H)
        if H is not None:
            h_uav, w_uav = 490, 490
            center_uav = np.array([[w_uav / 2, h_uav / 2]], dtype=np.float32)  
            center_uav[0][0] = center_uav[0][0]-10# 形状为 (1, 2)
            center_uav[0][1] = center_uav[0][1]-10# 形状为 (1, 2)
            # 将中心点坐标转换为齐次坐标
            center_uav_homogeneous = np.array([center_uav[0][0], center_uav[0][1], 1.0])  # 形状为 (3,)
            # 通过单应性矩阵进行变换
            center_tif_homogeneous = np.dot(H, center_uav_homogeneous)  # 形状为 (3,)
            # 归一化
            center_tif = center_tif_homogeneous[:2] / center_tif_homogeneous[2]  # 形状为 (2,)
            # center_tif = center_tif_homogeneous[:2] / 1 # 形状为 (2,)
            angle = extract_rotation_angle(H)
            print(f"旋转角度 (度): {angle}")
            # 打印结果
            print(f"无人机图像中心点在tif的位置：{center_tif}")
            pixel_x, pixel_y = center_tif
            tif_name = os.path.basename(img_tif)
            match = re.match(r"(\d+)_(\d+)_(\d+).tif", tif_name)
            if match:
                start_x = match.group(2)
                start_y = match.group(3)
                x_in_map = int(start_x) + pixel_x
                y_in_map = int(start_y) + pixel_y
                print(f"无人机图像中心点在地图上的位置：{x_in_map},{y_in_map}")
            try:
                lon, lat, x_geo, y_geo = pixel_to_geo_coordinates(x_in_map, y_in_map, geotransform)
                print(f"无人机图像中心点的经纬度：{lat}, {lon}")
                if n == 0:
                    x_origin, y_origin = x_geo, y_geo
                loc_file.write(f"{img_uav} {lon:.8f} {lat:.8f} {x_in_map:.8f} {y_in_map:.8f} {x_geo:.8f} {y_geo:.8f} {x_geo-x_origin:.10f} {y_origin-y_geo:.10f} {angle:.8f}\n")
                n+=1
            except Exception as e:
                print(f"无法计算无人机图像中心点的经纬度：{e}")
end = time.time()
print(f"Time: {end-start:.3f}s")

Found 429 image pairs.
UAV: uav/001.jpg - TIF: tif/278_1906_2356.tif
(320, 2)
[[1.33131993e+00 2.18065977e-02 8.64773178e+01]
 [2.67233215e-02 1.30766773e+00 4.44420738e+01]
 [7.36660877e-05 1.70919666e-05 1.00000000e+00]]
旋转角度 (度): 0.1067482057812912
无人机图像中心点在tif的位置：[396.0157705 350.5474455]
无人机图像中心点在地图上的位置：2302.0157705043293,2706.547445496444
无人机图像中心点的经纬度：34.24383741359542, 108.91089860854558
UAV: uav/002.jpg - TIF: tif/278_1906_2356.tif
(307, 2)
[[1.57393777e+00 1.87830985e-01 1.32710123e+01]
 [1.13561966e-01 1.52939808e+00 7.65768337e+00]
 [2.53652135e-04 2.77988758e-04 1.00000000e+00]]
旋转角度 (度): -1.3709406333613774
无人机图像中心点在tif的位置：[379.83211371 350.02296261]
无人机图像中心点在地图上的位置：2285.8321137091975,2706.0229626140426
无人机图像中心点的经纬度：34.24383857650447, 108.91085520059542
UAV: uav/003.jpg - TIF: tif/278_1906_2356.tif
(276, 2)
[[1.48508883e+00 9.62294936e-02 2.44780941e+01]
 [9.32084396e-02 1.46474218e+00 1.71355019e+01]
 [2.16309898e-04 1.77274182e-04 1.00000000e+00]]
旋转角度 (度): -0.0586791518

In [16]:
import cv2
from my_pkg.tools import parse_keyframe_file, draw_keyframe_trajectory, plot_traj_tif, draw_keyframe_elevation_trajectory
points_traj = []
with open("/home/lty/outputs/RealUAV/city1/loc_from_gt.txt", 'r') as loc_file:
    for line in loc_file:
        parts = line.strip().split()
        if len(parts) < 4:
            continue
        x_in_map = float(parts[3])   # 调整横坐标
        y_in_map = float(parts[4])  # 调整纵坐标
        print(f"{parts[0]}: {x_in_map}, {y_in_map}")
        points_traj.append((x_in_map, y_in_map))

    # 将点转换为整数坐标（像素坐标）
points_traj = [(int(x), int(y)) for x, y in points_traj]

# plot_traj_tif(
#     map_image_path="/home/lty/outputs/RealUAV/city1/gt_match(kf).png",
#     loc_file_path="/home/lty/outputs/RealUAV/city1/loc_elevation.txt",
#     output_image_path=output_dir/"gt_match_elevation().png",
#     scale_factor=1,
# )

# # 绘制关键帧的单独景象匹配结果
# map = cv2.imread("/home/lty/outputs/RealUAV/city1/gt_match(kf).png", cv2.IMREAD_COLOR)
# keyframe_mapping = parse_keyframe_file("/home/lty/outputs/RealUAV/city1/KeyFrameId.txt")
# map_with_traj = draw_keyframe_trajectory(map, points_traj, keyframe_mapping)
# cv2.imwrite(output_dir/"gt_match_elevation(kf).png", map_with_traj)
# # 绘制关键帧elevation的单独景象匹配结果
map = cv2.imread("/home/lty/outputs/RealUAV/city1/gt.png", cv2.IMREAD_COLOR)
keyframe_mapping = parse_keyframe_file("/home/lty/outputs/RealUAV/city1/KeyFrameId.txt")
map_with_traj = draw_keyframe_elevation_trajectory(map, points_traj, keyframe_mapping)
cv2.imwrite(output_dir/"gt_tiaozheng.png", map_with_traj)

uav/001.jpg: 0.0, 0.0
uav/002.jpg: 0.0, 0.0
uav/003.jpg: 0.0, 0.0
uav/004.jpg: 0.0, 0.0
uav/005.jpg: 0.0, 0.0
uav/006.jpg: 0.0, 0.0
uav/007.jpg: 0.0, 0.0
uav/008.jpg: 0.0, 0.0
uav/009.jpg: 0.0, 0.0
uav/010.jpg: 0.0, 0.0
uav/011.jpg: 0.0, 0.0
uav/012.jpg: 0.0, 0.0
uav/013.jpg: 0.0, 0.0
uav/014.jpg: 0.0, 0.0
uav/015.jpg: 0.0, 0.0
uav/016.jpg: 0.0, 0.0
uav/017.jpg: 0.0, 0.0
uav/018.jpg: 0.0, 0.0
uav/019.jpg: 0.0, 0.0
uav/020.jpg: 0.0, 0.0
uav/021.jpg: 0.0, 0.0
uav/022.jpg: 0.0, 0.0
uav/023.jpg: 0.0, 0.0
uav/024.jpg: 0.0, 0.0
uav/025.jpg: 0.0, 0.0
uav/026.jpg: 0.0, 0.0
uav/027.jpg: 0.0, 0.0
uav/028.jpg: 0.0, 0.0
uav/029.jpg: 0.0, 0.0
uav/030.jpg: 0.0, 0.0
uav/031.jpg: 0.0, 0.0
uav/032.jpg: 0.0, 0.0
uav/033.jpg: 0.0, 0.0
uav/034.jpg: 0.0, 0.0
uav/035.jpg: 0.0, 0.0
uav/036.jpg: 0.0, 0.0
uav/037.jpg: 0.0, 0.0
uav/038.jpg: 0.0, 0.0
uav/039.jpg: 0.0, 0.0
uav/040.jpg: 0.0, 0.0
uav/041.jpg: 0.0, 0.0
uav/042.jpg: 0.0, 0.0
uav/043.jpg: 0.0, 0.0
uav/044.jpg: 0.0, 0.0
uav/045.jpg: 0.0, 0.0
uav/046.jp

True

slam traj

In [25]:
#fusion traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj, draw_fusion_keyframe_traj
slam_traj_path = "/home/lty/outputs/RealUAV/city1/geoKFrame_proposed(slam).txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx  # 调整横坐标  seu need *0.2
        y_in_map = Pixely 
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"gt_match_elevation(kf).png", cv2.IMREAD_COLOR)
map_with_traj = draw_fusion_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"gt_match_elevation_proposed(kf).png", map_with_traj)

True

In [26]:
#slam traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj
slam_traj_path = "/home/lty/outputs/RealUAV/city1/geoKFrame_proposed(slam).txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx  # 调整横坐标  seu need *0.2
        y_in_map = Pixely 
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"gt_match_elevation_proposed(kf).png", cv2.IMREAD_COLOR)
map_with_traj = draw_slam_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"compare.png", map_with_traj)

True

In [17]:
# 打开 HDF5 文件
with h5py.File(features_path, 'r') as f:
    # 获取所有数据集的名称
    print("文件中的数据集和分组结构：")
    def print_structure(name, obj):
        """递归打印 HDF5 文件的层次结构"""
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name} - Shape: {obj.shape} - Type: {obj.dtype}")
    f.visititems(print_structure)

    # 示例：读取一个特定图像的特征
    example_image = "seu_uav/DJI_0218.JPG"  # 替换为您的图像名称
    if example_image in f:
        group = f[example_image]
        print(f"\n特定图像 '{example_image}' 的内容:")
        for key in group.keys():
            data = group[key][:]
            print(f"{key}: {data.shape} - {data.dtype}")
    else:
        print(f"图像 '{example_image}' 不存在于 HDF5 文件中。")

文件中的数据集和分组结构：
Group: tif
Group: tif/100_856_1006.tif
Dataset: tif/100_856_1006.tif/descriptors - Shape: (256, 1919) - Type: float16
Dataset: tif/100_856_1006.tif/image_size - Shape: (2,) - Type: int64
Dataset: tif/100_856_1006.tif/keypoints - Shape: (1919, 2) - Type: float16
Dataset: tif/100_856_1006.tif/scores - Shape: (1919,) - Type: float16
Group: tif/101_1006_1006.tif
Dataset: tif/101_1006_1006.tif/descriptors - Shape: (256, 1911) - Type: float16
Dataset: tif/101_1006_1006.tif/image_size - Shape: (2,) - Type: int64
Dataset: tif/101_1006_1006.tif/keypoints - Shape: (1911, 2) - Type: float16
Dataset: tif/101_1006_1006.tif/scores - Shape: (1911,) - Type: float16
Group: tif/102_1156_1006.tif
Dataset: tif/102_1156_1006.tif/descriptors - Shape: (256, 1901) - Type: float16
Dataset: tif/102_1156_1006.tif/image_size - Shape: (2,) - Type: int64
Dataset: tif/102_1156_1006.tif/keypoints - Shape: (1901, 2) - Type: float16
Dataset: tif/102_1156_1006.tif/scores - Shape: (1901,) - Type: float16
Gr